In [0]:
display(dbutils.fs.ls("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/,raw_data/,0,1792312284000
abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed_data/,transformed_data/,0,1792312294000


In [0]:
spark

In [0]:
athletes = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/athlete.csv")
coaches = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/coaches.csv")
entriesgender = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/EntriesGender.csv")
medals = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/Medals.csv")
teams = spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/raw_data/Teams.csv")

In [0]:
athletes.show()

+--------------------+--------------------+-------------------+
|          PersonName|             Country|         Discipline|
+--------------------+--------------------+-------------------+
|     AALERUD Katrine|              Norway|       Cycling Road|
|         ABAD Nestor|               Spain|Artistic Gymnastics|
|   ABAGNALE Giovanni|               Italy|             Rowing|
|      ABALDE Alberto|               Spain|         Basketball|
|       ABALDE Tamara|               Spain|         Basketball|
|           ABALO Luc|              France|           Handball|
|        ABAROA Cesar|               Chile|             Rowing|
|       ABASS Abobakr|               Sudan|           Swimming|
|    ABBASALI Hamideh|Islamic Republic ...|             Karate|
|       ABBASOV Islam|          Azerbaijan|          Wrestling|
|        ABBINGH Lois|         Netherlands|           Handball|
|         ABBOT Emily|           Australia|Rhythmic Gymnastics|
|       ABBOTT Monica|United States of .

In [0]:
athletes.printSchema()

root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)



In [0]:
coaches.show()

+--------------------+--------------------+-----------------+--------+
|                Name|             Country|       Discipline|   Event|
+--------------------+--------------------+-----------------+--------+
|     ABDELMAGID Wael|               Egypt|         Football|    NULL|
|           ABE Junya|               Japan|       Volleyball|    NULL|
|       ABE Katsuhiko|               Japan|       Basketball|    NULL|
|        ADAMA Cherif|       C�te d'Ivoire|         Football|    NULL|
|          AGEBA Yuya|               Japan|       Volleyball|    NULL|
|AIKMAN Siegfried ...|               Japan|           Hockey|     Men|
|       AL SAADI Kais|             Germany|           Hockey|     Men|
|       ALAMEDA Lonni|              Canada|Baseball/Softball|Softball|
|     ALEKNO Vladimir|Islamic Republic ...|       Volleyball|     Men|
|     ALEKSEEV Alexey|                 ROC|         Handball|   Women|
|ALLER CARBALLO Ma...|               Spain|       Basketball|    NULL|
|     

In [0]:
coaches.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)



In [0]:

# Calculate the average number of entries by gender for each discipline
average_entries_by_gender = entriesgender.withColumn(
    'Avg_Female', entriesgender['Female'] / entriesgender['Total']
).withColumn(
    'Avg_Male', entriesgender['Male'] / entriesgender['Total']
)
average_entries_by_gender.show()

+--------------------+------+----+-----+-------------------+-------------------+
|          Discipline|Female|Male|Total|         Avg_Female|           Avg_Male|
+--------------------+------+----+-----+-------------------+-------------------+
|      3x3 Basketball|    32|  32|   64|                0.5|                0.5|
|             Archery|    64|  64|  128|                0.5|                0.5|
| Artistic Gymnastics|    98|  98|  196|                0.5|                0.5|
|   Artistic Swimming|   105|   0|  105|                1.0|                0.0|
|           Athletics|   969|1072| 2041| 0.4747672709456149| 0.5252327290543851|
|           Badminton|    86|  87|  173|0.49710982658959535| 0.5028901734104047|
|   Baseball/Softball|    90| 144|  234|0.38461538461538464| 0.6153846153846154|
|          Basketball|   144| 144|  288|                0.5|                0.5|
|    Beach Volleyball|    48|  48|   96|                0.5|                0.5|
|              Boxing|   102

In [0]:

coaches.repartition(1).write.mode("overwrite").option("header","true").csv("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed-data/coaches")
entriesgender.repartition(1).write.mode("overwrite").option("header","true").csv("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed-data/entriesgender")
medals.repartition(1).write.mode("overwrite").option("header","true").csv("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed-data/medals")
teams.repartition(1).write.mode("overwrite").option("header","true").csv("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed-data/teams")
    

In [0]:
athletes.repartition(1).write.mode("overwrite").option("header",'true').csv("abfss://tokyoolympicdata@olympicdatastoragenew.dfs.core.windows.net/transformed-data/athletes")